# Setting up GB1 Data

MAVENN documentation provides a tutorial where they use Otwinowski model parameters to generate ddG values for benchmarking their own model. This follows the same procedure, but generates all single and double mutants for input into the experiment simulator. Then, it calculates probabilities of each mutation based on the average number of nucleotides required to produce that mutation to assign counts.

In [2]:
import pandas as pd
import numpy as np
import itertools

def generate_full_gb1_library(path_bind, path_fold):
    """
    Generates a DataFrame of all Single and Double mutants for GB1,
    calculating ddG values based on the Otwinowski additive model.
    """
    print(f"1. Loading raw data from {path_bind} and {path_fold}...")
    
    # Load Data
    try:
        df_bind = pd.read_csv(path_bind, index_col=0)
        df_fold = pd.read_csv(path_fold, index_col=0)
    except FileNotFoundError:
        print("ERROR: CSV files not found. Please ensure they are in the folder.")
        return None

    # --- ORIENTATION CHECK ---
    # Ensure Index = Position (Int), Columns = Amino Acids (Str)
    if 'A' in df_bind.index: 
        print("   -> Transposing Binding Matrix...")
        df_bind = df_bind.T
    if 'A' in df_fold.index:
        print("   -> Transposing Folding Matrix...")
        df_fold = df_fold.T
        
    df_bind.index = df_bind.index.astype(int)
    df_fold.index = df_fold.index.astype(int)

    # --- SETUP ---
    # GB1 Sequence (Positions 2-56)
    wt_seq = "QYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE"
    # Map index 0..54 -> Position 2..56
    pos_map = {i: i+2 for i in range(len(wt_seq))}
    
    alphabet = list("ACDEFGHIKLMNPQRSTVWY")
    
    # Pre-calculate Single Mutant Effects (ddG = Mut - WT)
    # This dictionary will store: { pos_idx: { mut_aa: (ddG_bind, ddG_fold) } }
    print("2. Normalizing single mutant effects...")
    singles_map = {}
    
    for i, wt_aa in enumerate(wt_seq):
        pos = pos_map[i]
        
        # Skip if position not in data
        if pos not in df_bind.index: continue
            
        singles_map[pos] = {}
        
        # Get WT Energies
        wt_b = df_bind.loc[pos, wt_aa]
        wt_f = df_fold.loc[pos, wt_aa]
        
        for mut_aa in alphabet:
            if mut_aa == wt_aa: continue
            
            # Calculate ddG
            ddg_b = df_bind.loc[pos, mut_aa] - wt_b
            ddg_f = df_fold.loc[pos, mut_aa] - wt_f
            
            singles_map[pos][mut_aa] = (ddg_b, ddg_f)

    # --- GENERATION ---
    records = []
    
    # A. Wild Type
    records.append(['', 0.0, 0.0])
    
    # B. Single Mutants
    print("3. Generating Single Mutants...")
    sorted_positions = sorted(singles_map.keys())
    
    for pos in sorted_positions:
        wt_aa = wt_seq[pos-2]
        for mut_aa, (db, df) in singles_map[pos].items():
            name = f"{wt_aa}{pos}{mut_aa}"
            records.append([name, df, db]) # Note: Order is ddG_fold, ddG_bind

    # C. Double Mutants
    print("4. Generating Double Mutants (this takes a moment)...")
    
    # Iterate through all unique pairs of positions
    for i in range(len(sorted_positions)):
        pos1 = sorted_positions[i]
        wt_aa1 = wt_seq[pos1-2]
        
        for j in range(i + 1, len(sorted_positions)):
            pos2 = sorted_positions[j]
            wt_aa2 = wt_seq[pos2-2]
            
            # Get all mutations at pos1 and pos2
            muts1 = singles_map[pos1].items()
            muts2 = singles_map[pos2].items()
            
            # Cartesian product of mutations at these two positions
            for (m1, (db1, df1)), (m2, (db2, df2)) in itertools.product(muts1, muts2):
                
                # Construct Name: "Q2A Y3C"
                name = f"{wt_aa1}{pos1}{m1} {wt_aa2}{pos2}{m2}"
                
                # Additive Model: Sum of effects
                total_bind = db1 + db2
                total_fold = df1 + df2
                
                records.append([name, total_fold, total_bind])

    # --- FINALIZE ---
    print("5. Building DataFrame...")
    df = pd.DataFrame(records, columns=["aa_substitutions", "ddG_fold", "ddG_bind"])
    
    print(f"Done! Generated {len(df):,} variants.")
    return df

In [ ]:
BIND_FILE = "otwinowski_gb_data.csv"
FOLD_FILE = "otwinowski_gf_data.csv"

gb1_library = generate_full_gb1_library(BIND_FILE, FOLD_FILE)
gb1_library

1. Loading raw data from otwinowski_gb_data.csv and otwinowski_gf_data.csv...
   -> Transposing Binding Matrix...
   -> Transposing Folding Matrix...
2. Normalizing single mutant effects...
3. Generating Single Mutants...
4. Generating Double Mutants (this takes a moment)...
5. Building DataFrame...
Done! Generated 537,131 variants.


,aa_substitutions,ddG_fold,ddG_bind
0,,0.000000,0.000000
1,Q2A,1.977038,-0.354784
2,Q2C,1.926444,0.097025
3,Q2D,2.676374,0.199019
4,Q2E,1.126964,0.185110
...,...,...,...
537126,T55Y E56S,2.206998,1.931861
537127,T55Y E56T,2.441320,1.787153
537128,T55Y E56V,2.289906,2.214742
537129,T55Y E56W,1.240114,2.007181


In [22]:
import pandas as pd
import numpy as np
from collections import defaultdict

# --- 1. GENETIC CODE CONSTANTS ---
# Standard Codon Table (DNA)
CODON_TABLE = {
    'A': ['GCT', 'GCC', 'GCA', 'GCG'],
    'C': ['TGT', 'TGC'],
    'D': ['GAT', 'GAC'],
    'E': ['GAA', 'GAG'],
    'F': ['TTT', 'TTC'],
    'G': ['GGT', 'GGC', 'GGA', 'GGG'],
    'H': ['CAT', 'CAC'],
    'I': ['ATT', 'ATC', 'ATA'],
    'K': ['AAA', 'AAG'],
    'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
    'M': ['ATG'],
    'N': ['AAT', 'AAC'],
    'P': ['CCT', 'CCC', 'CCA', 'CCG'],
    'Q': ['CAA', 'CAG'],
    'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
    'T': ['ACT', 'ACC', 'ACA', 'ACG'],
    'V': ['GTT', 'GTC', 'GTA', 'GTG'],
    'W': ['TGG'],
    'Y': ['TAT', 'TAC'],
    '_': ['TAA', 'TAG', 'TGA'] # Stop codons (optional)
}

def get_min_nt_distance(aa_wt, aa_mut):
    """
    Calculates the minimum number of nucleotide changes required
    to change from aa_wt to aa_mut.
    """
    if aa_wt == aa_mut:
        return 0
    
    # Get all codon possibilities
    codons_wt = CODON_TABLE.get(aa_wt, [])
    codons_mut = CODON_TABLE.get(aa_mut, [])
    
    if not codons_wt or not codons_mut:
        return 3 # Default to max distance if unknown
    
    min_dist = 3
    
    # Brute force compare all codon pairs (computationally cheap: max 6x6=36 pairs)
    for c1 in codons_wt:
        for c2 in codons_mut:
            # Hamming distance calculation
            dist = sum(1 for n1, n2 in zip(c1, c2) if n1 != n2)
            if dist < min_dist:
                min_dist = dist
                if min_dist == 1: 
                    return 1 # Optimization: Can't get better than 1
                    
    return min_dist

def assign_library_counts(df, total_counts=1e8, mutation_rate=0.01, random_seed=None):
    """
    Assigns simulated read counts to a library based on mutational probability.
    
    Args:
        df (pd.DataFrame): Must contain 'aa_substitutions' column.
        total_counts (float): The total size of the physical library (e.g. 1e8 cells).
        mutation_rate (float): Probability factor per nucleotide mutation. 
                               Higher = more mutants, Lower = dominant Wild Type.
        random_seed (int): For reproducibility.
        
    Returns:
        pd.DataFrame: Original df with new 'n_cells_initial' column.
    """
    if random_seed is not None:
        np.random.seed(random_seed)
        
    print(f"Generating counts for {len(df):,} variants (Total: {total_counts:.0e})...")
    
    # 1. Calculate nucleotide distances for every variant
    # This determines the "weight" or likelihood of the variant arising.
    weights = []
    
    for subs in df['aa_substitutions']:
        if pd.isna(subs) or subs == '':
            # Wild Type
            weights.append(1.0) # Base weight for WT
            continue
            
        # Parse "Q2A:Y3C" -> ["Q2A", "Y3C"]
        mutations = subs.split(':')
        
        total_dist = 0
        for mut in mutations:
            # Parse "Q2A" -> wt=Q, mut=A
            wt_aa = mut[0]
            mut_aa = mut[-1]
            total_dist += get_min_nt_distance(wt_aa, mut_aa)
            
        # Probability Model: P ~ (mutation_rate) ^ distance
        # Example: 1 change = 0.01, 2 changes = 0.0001
        # This creates the realistic exponential drop-off seen in epPCR.
        weight = (mutation_rate) ** total_dist
        weights.append(weight)
        
    weights = np.array(weights)
    
    # 2. Normalize to Probabilities
    # The Wild Type (weight 1.0) will likely dominate the probability mass
    probs = weights / weights.sum()
    
    # 3. Multinomial Sampling (Poisson-like statistics constrained to total sum)
    # This handles the "Shot Noise" - rare variants might get 0 counts.
    # We cast to int64 to avoid overflow with large numbers
    counts = np.random.multinomial(n=int(total_counts), pvals=probs)
    
    df['count'] = counts
    
    # --- QC REPORT ---
    n_zeros = (counts == 0).sum()
    wt_count = df.loc[df.aa_substitutions == '', 'count'].sum()
    print(f"  -> WT Count: {wt_count:,.0f} ({wt_count/total_counts:.1%})")
    print(f"  -> Variants lost (Count=0): {n_zeros} / {len(df)}")
    
    return df

In [23]:
gb1_library = assign_library_counts(
    gb1_library, 
    total_counts=1e8, 
    mutation_rate=0.21
)

gb1_library

Generating counts for 537,131 variants (Total: 1e+08)...
  -> WT Count: 1,227 (0.0%)
  -> Variants lost (Count=0): 0 / 537131


,aa_substitutions,ddG_fold,ddG_bind,count
0,,0.000000,0.000000,1227
1,Q2A,1.977038,-0.354784,47
2,Q2C,1.926444,0.097025,12
3,Q2D,2.676374,0.199019,54
4,Q2E,1.126964,0.185110,263
...,...,...,...,...
537126,T55Y E56S,2.206998,1.931861,238
537127,T55Y E56T,2.441320,1.787153,1216
537128,T55Y E56V,2.289906,2.214742,63
537129,T55Y E56W,1.240114,2.007181,51


In [24]:
gb1_library.to_csv("gb1_full_library_ddG.csv", index=False)